<!--- sf-header --->
<table align="left">
<tr>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Fstatmike%2Fscale-forecasting%2Fmain%2Fnotebooks%2F07_scale_review.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Colab Enterprise logo">
      <br>Run in<br>Colab Enterprise
    </a>
  </td>
</tr>
</table>
<br clear="left"/>

> **Run in Colab Enterprise:** click the badge to import this notebook, pick a runtime, and
> **Run all**. The Terraform-deployed templates already carry the `SF_*` run identity in their env,
> so there's no environment cell to fill in. Runs on the **`sf-main`** runtime template (Python 3.11). See
> [`docs/notebook_runtimes.md`](https://github.com/statmike/scale-forecasting/blob/main/docs/notebook_runtimes.md)
> for the per-notebook template mapping and the headless acceptance harness.


# 07 · Scale review — runtimes and the family DAG, one board

The cross-run comparison. The other notebooks each run **one** thing; this one *reviews many at once*. Point it at the `run_id`s you already ran and it renders them side by side:

- **runtime parity** — the *same* models on **Spark** vs **Ray** (`explode_100k` vs `ray_100k`): same unit of work, same answers, different engine, so the difference is wall-clock and provisioning overhead.
- **the family DAG** — one config (`all_families_100k`) fanned into a **job per model family** (statistical / ml / deep-learning / native), each on its resolved runtime under one `run_id`. `v_run_jobs` shows where each family landed.

It reads three registry views: the **scaling-and-efficiency** story from `v_run_summary` (wall-clock, provisioning overhead, DCU), the **per-family placement** from `v_run_jobs`, and the **accuracy** story from `v_model_leaderboard` (which model won, and how fast it fit).

It **runs nothing** — no config, no `main.run`. It only reads the registry views, so it's cheap and re-runnable while the CLI-submitted 100k runs finish.

## Get the code (cloud runtimes only)

On a cloud notebook (Colab Enterprise, Vertex Workbench) this clones or updates the repo so you're on the latest `src/`. **Skip it in a local clone** — it's a no-op guarded on the package already being importable.

In [1]:
# Cloud bootstrap: clone + install the LOCKED dependency set (uv.lock) so
# `import scale_forecasting` resolves against the exact versions every other surface runs.
# Uses uv (Colab ships it; we install it if missing) to install the frozen lock into a PRIVATE
# prefix, then puts that prefix + src/ first on sys.path. Harmless locally — if it already imports,
# no-op.
import importlib.util
import os
import shutil
import subprocess
import sys

REPO_URL = os.environ.get("SF_REPO_URL", "https://github.com/statmike/scale-forecasting.git")
REPO_DIR = os.environ.get("SF_REPO_DIR", "scale-forecasting")

EXTRAS = []  # this notebook needs only the core

if importlib.util.find_spec("scale_forecasting") is None:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    uv = shutil.which("uv")
    if uv is None:  # Colab Enterprise ships uv; install it if this runtime doesn't
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
        uv = shutil.which("uv") or "uv"
    # Resolve the checked-in lock to a requirements file (no re-resolve), then install EXACTLY
    # that set into this kernel — the same versions the container + packed-venv are built from.
    reqs = os.path.abspath(os.path.join(REPO_DIR, "colab-requirements.txt"))
    extra_flags = [f for e in EXTRAS for f in ("--extra", e)]
    subprocess.run(
        [uv, "export", "--frozen", "--no-emit-project", "--no-hashes", "--no-dev", *extra_flags,
         "-o", reqs],
        cwd=REPO_DIR, check=True,
    )
    # Install the LOCKED set into a PRIVATE directory (not the runtime's system site-packages),
    # then put it FIRST on sys.path. Managed images (Colab Enterprise, Vertex) ship numpy 2.x that
    # can't be cleanly downgraded in place: uv skips the version-satisfied packages it can't fully
    # uninstall, so numpy-2 .so files end up beside numpy-1 python and `import numpy` dies with
    # "dtype size changed, Expected 96 ... got 88". Installing to a separate prefix and SHADOWING the
    # base packages (never touching them) gives this kernel a clean, self-consistent numpy 1.26.4 —
    # the same set every other surface runs — with zero risk of a mixed install.
    target = os.path.abspath(os.path.join(REPO_DIR, ".colab-deps"))
    subprocess.run(
        [uv, "pip", "install", "--python", sys.executable, "--target", target, "-r", reqs],
        check=True,
    )
    # src/ carries our package; `target` carries its locked deps. Both go ahead of the base image's
    # site-packages so imports resolve to the versions we installed, not the runtime's.
    sys.path.insert(0, target)
    sys.path.insert(0, os.path.join(REPO_DIR, "src"))

## Resolve the deployment (live GCP)

`Settings.resolve()` reads the `SF_*` environment — the *same* identity every writer uses. Required: `SF_PROJECT_ID`, `SF_CONNECTION`, `SF_WAREHOUSE_URI`; `SF_DATASET_ID`/`SF_REGION` default. This notebook only *reads* the `v_run_summary` / `v_run_jobs` / `v_model_leaderboard` views.

In [2]:
from google.cloud import bigquery

from scale_forecasting.settings import Settings

settings = Settings.resolve()
client = bigquery.Client(project=settings.project_id)
DATASET = settings.dataset_ref
print("deployment:", DATASET, "region:", settings.region)

deployment: statmike-scale-forecasting.scale_forecasting region: us-central1


## Parameters — the run_ids to compare

Fill in the `run_id`s printed by the Act 1 CLI-submitted 100k runs (or the notebook demos at any scale). Leave a value `None` to skip that run (the review just omits it).

> The keys are labels for the comparison: `spark` and `ray` are the *same models on different runtimes* (`explode_100k` vs `ray_100k`), and `all-families` is the one config fanned into a job per family (`all_families_100k`). Because `run_id` is a deterministic digest of the config, the shipped defaults below already match the unchanged configs — paste yours only if you overrode a config.

Find recent ids with the discovery cell below if you don't have them handy.

In [3]:
# === Parameters — edit me ===============================================
# One run_id per label (None → skip). Deterministic defaults match the shipped configs.
RUN_IDS = {
    "spark": "explode-100k-1df567200b77",         # explode_100k.json — statistical+ml on Spark
    "ray": "ray-100k-afca38f63f92",               # ray_100k.json — same models, Ray runtime
    "all-families": "all-families-100k-050eee3e0a6b",  # all_families_100k.json — a job per family
}
# ========================================================================

runs = {name: rid for name, rid in RUN_IDS.items() if rid}
print("comparing", len(runs), "run(s):")
for name, rid in runs.items():
    print(f"  {name:<14} {rid}")

comparing 3 run(s):
  spark          explode-100k-1df567200b77
  ray            ray-100k-afca38f63f92
  all-families   all-families-100k-050eee3e0a6b


## (Optional) Discover recent run_ids

Don't have the ids? This lists the most recent completed runs with their `python_runtime` and scale, so you can copy the right `run_id` into the cell above. Purely a convenience — skip it if you already pasted them.

In [4]:
import pandas as pd

recent = client.query(
    f"SELECT run_id, created_at, status, python_runtime, n_series, n_models "
    f"FROM `{DATASET}.v_run_summary` "
    f"ORDER BY created_at DESC LIMIT 25"
).result().to_dataframe()
recent

,run_id,created_at,status,python_runtime,n_series,n_models
0,nb02-bq-native-1788324063-c598175571d2,2026-09-02 04:41:21.167911+00:00,COMPLETED,spark,100,2
1,wave10-ray-availability-probe-b352a2a2cb54,2026-09-02 04:25:20.815762+00:00,FAILED,ray,100,3
2,wave10-ray-100k-demofleet-8e102d25e409,2026-09-02 04:18:15.740549+00:00,FAILED,ray,100000,4
3,wave10-ray-100k-stdhead-2f8657b7886b,2026-09-02 04:11:11.428420+00:00,FAILED,ray,100000,4
4,wave10-ray-100k-fleet10-22d18ed400d3,2026-09-02 04:03:48.016336+00:00,FAILED,ray,100000,4
5,ray-100k-dcc77a9d1e9b,2026-09-02 03:51:12.809134+00:00,FAILED,ray,100000,4
6,ops-droprun-8575bf67-f997cfa4d0e7,2026-09-02 03:38:59.304283+00:00,RUNNING,spark,1,1
7,wave-62-mixed-runtimes-cpu-a7d04b6a9c8e,2026-09-02 01:38:00.968582+00:00,PARTIAL,spark,100,5
8,wave-61-shared-ray-cpu-6ea5b25ffe39,2026-09-02 01:37:48.223065+00:00,FAILED,ray,100,3
9,wave-68-level-shift-on-800462340da5,2026-09-02 01:00:43.208436+00:00,COMPLETED,spark,100,1


## Scaling + efficiency — `v_run_summary`, one row per run

The whole efficiency story in one table: `runtime_seconds` (the engine's own compute), `total_wall_s` (including cluster stand-up), `overhead_seconds` / `overhead_fraction` (provisioning tax — amortizes as `n_series` grows), and `dcu_milli_seconds` (the Dataproc cost proxy). Labeled by run so the Spark and Ray runs of the same models line up against each other.

In [5]:
def summaries(runs):
    """One v_run_summary row per run, labeled with the run name."""
    frames = []
    for name, rid in runs.items():
        df = client.query(
            f"SELECT * FROM `{DATASET}.v_run_summary` WHERE run_id=@run_id",
            job_config=bigquery.QueryJobConfig(
                query_parameters=[bigquery.ScalarQueryParameter("run_id", "STRING", rid)]
            ),
        ).result().to_dataframe()
        df.insert(0, "run", name)
        frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


summary = summaries(runs)
summary

,run,run_id,created_at,status,python_runtime,n_series,n_models,backtest_on,runtime_seconds,total_wall_s,overhead_seconds,overhead_fraction,executor_instances,executor_cores,max_executors,executor_memory,executor_memory_overhead,dcu_milli_seconds,runtime_version,sizing


## The showpiece — compute vs provisioning overhead, per run

A stacked bar per run: `runtime_seconds` (real forecasting work) plus `overhead_seconds` (cluster stand-up) = `total_wall_s`. At 100k series the compute bar should dominate. The `spark` and `ray` runs are the *same models*, so this is the runtime comparison — where each engine spends its time; `all-families` adds the deep-learning and native families, so its wall-clock is set by its *slowest* family.

In [6]:
import matplotlib.pyplot as plt

if not summary.empty:
    plot_df = summary.fillna({"runtime_seconds": 0, "overhead_seconds": 0})
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(plot_df["run"], plot_df["runtime_seconds"], label="runtime_seconds (compute)",
           color="#1f77b4")
    ax.bar(plot_df["run"], plot_df["overhead_seconds"], bottom=plot_df["runtime_seconds"],
           label="overhead_seconds (provisioning)", color="#ff7f0e")
    ax.set_ylabel("seconds")
    ax.set_title("wall-clock by run — compute vs provisioning")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No run_ids set — fill in RUN_IDS above.")

No run_ids set — fill in RUN_IDS above.


## Per-family placement — `v_run_jobs`

Where each run's work actually ran. `v_run_jobs` is one row per `(run_id, family)` (plus the ensemble node): the `family`, the `runtime` it resolved to (Spark / Ray / BigQuery), its `hardware` (cpu / gpu), and its `runtime_seconds`. For `all-families` this is the DAG made concrete — statistical/ml on the Python runtime, deep-learning on a GPU, native in BigQuery, all under one `run_id`. Sorted so each run's slowest family (its wall-clock driver) is on top.

In [7]:
def family_jobs(runs):
    """Every run's v_run_jobs rows stacked — one row per (run_id, family), plus the ensemble node."""
    frames = []
    for name, rid in runs.items():
        df = client.query(
            f"SELECT family, runtime, hardware, status, runtime_seconds "
            f"FROM `{DATASET}.v_run_jobs` WHERE run_id=@run_id",
            job_config=bigquery.QueryJobConfig(
                query_parameters=[bigquery.ScalarQueryParameter("run_id", "STRING", rid)]
            ),
        ).result().to_dataframe()
        df.insert(0, "run", name)
        frames.append(df)
    out = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    return out.sort_values(["run", "runtime_seconds"], ascending=[True, False]) if not out.empty else out


jobs = family_jobs(runs)
jobs

,run,family,runtime,hardware,status,runtime_seconds


## Accuracy — every model, every run, one leaderboard

All runs' `v_model_leaderboard` rows stacked, labeled by run. `worker.run_cell` is identical across engines, so a model's `mean_wape` should be consistent whether it ran under Spark or Ray — the runtime changes *speed*, not *answers*. Any drift here is a signal, not a feature. `median_fit_seconds` exposes each model's per-cell fit time.

In [8]:
def leaderboards(runs):
    """Stack every run's v_model_leaderboard rows, labeled by run."""
    frames = []
    for name, rid in runs.items():
        df = client.query(
            f"SELECT model_type, compute_engine, n_cells, no_artifact_rate, "
            f"median_fit_seconds, mean_wape, mean_mae "
            f"FROM `{DATASET}.v_model_leaderboard` WHERE run_id=@run_id",
            job_config=bigquery.QueryJobConfig(
                query_parameters=[bigquery.ScalarQueryParameter("run_id", "STRING", rid)]
            ),
        ).result().to_dataframe()
        df.insert(0, "run", name)
        frames.append(df)
    out = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    return out.sort_values(["model_type", "run"]) if not out.empty else out


board = leaderboards(runs)
board

,run,model_type,compute_engine,n_cells,no_artifact_rate,median_fit_seconds,mean_wape,mean_mae


## Same model, same answer — accuracy parity across runtimes

`mean_wape` for each model, grouped by run. Bars for the *same* model should sit at nearly the same height across the Spark and Ray runs — the visual proof that the runtime is an execution detail, not a modeling one. (Models that only exist in one run, e.g. the BigQuery natives in `all-families`, appear once.)

In [9]:
if board.empty:
    print("No leaderboard rows — fill in RUN_IDS above.")
elif board["mean_wape"].notna().any():
    pivot = board.pivot_table(index="model_type", columns="run", values="mean_wape")
    ax = pivot.plot(kind="bar", figsize=(11, 4))
    ax.set_ylabel("mean WAPE (lower is better)")
    ax.set_title("same model, same answer — mean_wape by run")
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.show()
else:
    # mean_wape is populated only when a backtest ran; the 100k scale runs turn backtest OFF for
    # speed, so every mean_wape is NULL and there is nothing to plot. Expected at scale — the
    # scaling/efficiency and per-family panels above are the showpiece; accuracy parity is a
    # small-scale story (turn backtest on in a demo config to populate it). See docs/output_schemas.md.
    print("mean_wape is all-NULL for these runs (backtest off — the 100k default).")
    print("Nothing to plot; see the scaling + efficiency and per-family panels above.")

No leaderboard rows — fill in RUN_IDS above.
